In [6]:
import numpy as np
import pandas as pd

In [ ]:
import requests, pandas as pd, time

API_KEY = "RDek4FNrRVhleQiaJqnvjPTcm9PKUJgOWsb8hJl3"
BASE_URL = "https://api.eia.gov/v2/electricity/rto/fuel-type-data/data/"

params = {
    "api_key": API_KEY,
    "frequency": "hourly",
    "data[0]": "value",
    "facets[respondent][]": "CISO",
    "start": "2012-01-01T00",
    "end": "2012-12-31T00",
    "sort[0][column]": "period",
    "sort[0][direction]": "desc",
    "length": 5000,
    "offset": 0,
}

all_rows = []
while True:
    r = requests.get(BASE_URL, params=params)
    data = r.json()["response"]
    rows = data["data"]
    all_rows.extend(rows)
    print(f"Fetched {len(all_rows)} / {data['total']}")
    if len(all_rows) >= int(data["total"]):
        break
    params["offset"] += 5000
    time.sleep(0.5)  # be polite to the API

df = pd.DataFrame(all_rows)
df.to_csv("caiso_2025_hourly.csv", index=False)

Fetched 5000 / 70248
Fetched 10000 / 70248
Fetched 15000 / 70248
Fetched 20000 / 70248
Fetched 25000 / 70248
Fetched 30000 / 70248
Fetched 35000 / 70248
Fetched 40000 / 70248
Fetched 45000 / 70248
Fetched 50000 / 70248
Fetched 55000 / 70248
Fetched 60000 / 70248
Fetched 65000 / 70248
Fetched 70000 / 70248
Fetched 70248 / 70248


In [4]:
df_pivot = (
    df
    .pivot_table(index="period", columns="fueltype", values="value", aggfunc="sum")
    .rename(columns=df.set_index("fueltype")["type-name"].to_dict())  # optional: use full names
)

In [5]:
df.to_csv("caiso_2025_hourly_raw.csv", index=False)
df_pivot.to_csv("caiso_2025_hourly.csv")

In [20]:
import requests

rtos = ["CISO", "ERCO", "ISNE", "MISO", "NYIS", "PJM", "SWPP", "BPAT"]

BASE_URL = "https://api.eia.gov/v2/electricity/rto/fuel-type-data/data/"

def available_years(rto, start_year=2019, end_year=2026):
    years = []

    for year in range(start_year, end_year + 1):
        params = {
            "api_key": API_KEY,
            "frequency": "hourly",
            "data[0]": "value",
            "facets[respondent][]": rto,
            "start": f"{year}-01-01T00",
            "end": f"{year + 1}-01-01T00",
            "length": 1,
        }

        r = requests.get(BASE_URL, params=params)
        response = r.json()["response"]

        if int(response["total"]) > 0:
            years.append(year)

        print(f"{rto}: {year} → {response['total']} records")

    return years

for rto in rtos:
    print(f"\nStarting {rto}\n===================")
    y = available_years(rto)
    print(y)


Starting CISO
CISO: 2019 → 65681 records
CISO: 2020 → 58902 records
CISO: 2021 → 70088 records
CISO: 2022 → 70088 records
CISO: 2023 → 70042 records
CISO: 2024 → 70075 records
CISO: 2025 → 70464 records
CISO: 2026 → 54711 records
[2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

Starting ERCO
ERCO: 2019 → 61327 records
ERCO: 2020 → 61495 records
ERCO: 2021 → 61327 records
ERCO: 2022 → 61327 records
ERCO: 2023 → 61327 records
ERCO: 2024 → 64157 records
ERCO: 2025 → 78098 records
ERCO: 2026 → 48616 records
[2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

Starting ISNE
ISNE: 2019 → 70088 records
ISNE: 2020 → 70280 records
ISNE: 2021 → 70088 records
ISNE: 2022 → 70088 records
ISNE: 2023 → 70088 records
ISNE: 2024 → 73209 records
ISNE: 2025 → 94499 records
ISNE: 2026 → 61559 records
[2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

Starting MISO
MISO: 2019 → 61327 records
MISO: 2020 → 61495 records
MISO: 2021 → 61351 records
MISO: 2022 → 61327 records
MISO: 2023 → 61327 records
MISO: 2024

In [21]:
import requests, pandas as pd, time

API_KEY = "RDek4FNrRVhleQiaJqnvjPTcm9PKUJgOWsb8hJl3"
BASE_URL = "https://api.eia.gov/v2/electricity/rto/fuel-type-data/data/"

rtos = ["CISO", "ERCO", "ISNE", "MISO", "NYIS", "PJM", "SWPP", "BPAT"]
years = np.arange(2019,2025)

for year in years:
    print(f"\n{year}\n==============================")
    for rto in rtos:
        params = {
            "api_key": API_KEY,
            "frequency": "hourly",
            "data[0]": "value",
            "facets[respondent][]": rto,
            "start": f"{year}-01-01T00",
            "end": f"{year+1}-01-01T00",
            "sort[0][column]": "period",
            "sort[0][direction]": "desc",
            "length": 5000,
            "offset": 0,
        }

        all_rows = []
        while True:
            r = requests.get(BASE_URL, params=params)
            data = r.json()["response"]
            rows = data["data"]
            all_rows.extend(rows)
            #print(f"{rto}: Fetched {len(all_rows)} / {data['total']}")
            if len(all_rows) >= int(data["total"]):
                break
            params["offset"] += 5000
            time.sleep(0.5)

        df = pd.DataFrame(all_rows)

        df_pivot = (
            df
            .pivot_table(index="period", columns="fueltype", values="value", aggfunc="sum")
            .rename(columns=df.set_index("fueltype")["type-name"].to_dict())  # optional: use full names
        )
        
        df.to_csv(f"hourly_fuel_mix/{rto.lower()}_{year}_hourly_raw.csv", index=False)
        df_pivot.to_csv(f"hourly_fuel_mix/{rto.lower()}_{year}_hourly.csv")

        print(f"Saved {rto} {year}")


2019
Saved CISO 2019
Saved ERCO 2019
Saved ISNE 2019
Saved MISO 2019
Saved NYIS 2019
Saved PJM 2019
Saved SWPP 2019
Saved BPAT 2019

2020
Saved CISO 2020
Saved ERCO 2020
Saved ISNE 2020
Saved MISO 2020
Saved NYIS 2020
Saved PJM 2020
Saved SWPP 2020
Saved BPAT 2020

2021
Saved CISO 2021
Saved ERCO 2021
Saved ISNE 2021
Saved MISO 2021
Saved NYIS 2021
Saved PJM 2021
Saved SWPP 2021
Saved BPAT 2021

2022
Saved CISO 2022
Saved ERCO 2022
Saved ISNE 2022
Saved MISO 2022
Saved NYIS 2022
Saved PJM 2022
Saved SWPP 2022
Saved BPAT 2022

2023
Saved CISO 2023
Saved ERCO 2023
Saved ISNE 2023
Saved MISO 2023
Saved NYIS 2023
Saved PJM 2023
Saved SWPP 2023
Saved BPAT 2023

2024
Saved CISO 2024
Saved ERCO 2024
Saved ISNE 2024
Saved MISO 2024
Saved NYIS 2024
Saved PJM 2024
Saved SWPP 2024
Saved BPAT 2024


In [8]:
rtos = ["CISO", "ERCO", "ISNE", "MISO", "NYIS", "PJM", "SWPP", "BPAT"]

for rto in rtos:
    in_path = f"{rto.lower()}_2025_hourly.csv"
    df = pd.read_csv(in_path,index_col=0,header=0)

    df_pivot = (
        df
        .pivot_table(index="period", columns="fueltype", values="value", aggfunc="sum")
        .rename(columns=df.set_index("fueltype")["type-name"].to_dict())  # optional: use full names
    )
    
    df.to_csv(f"{rto.lower()}_2025_hourly_raw.csv", index=False)
    df_pivot.to_csv(f"{rto.lower()}_2025_hourly.csv")

In [24]:
rto_timezones = {
    "CISO": "America/Los_Angeles",  # California
    "ERCO": "America/Chicago",       # ERCOT (Texas)
    "ISNE": "America/New_York",      # ISO New England
    "MISO": "America/Chicago",       # Midcontinent (spans zones, Chicago is standard)
    "NYIS": "America/New_York",      # New York
    "PJM":  "America/New_York",      # PJM
    "SWPP": "America/Chicago",       # SPP (Southwest Power Pool)
    "BPAT": "America/Los_Angeles",   # Bonneville Power (Pacific Northwest)
}

for year in np.append(years,2025):
    for rto in rtos:
        in_path = f"hourly_fuel_mix/{rto.lower()}_{year}_hourly.csv"
        df = pd.read_csv(in_path, header=0)
        df["period"] = pd.to_datetime(df["period"], utc=True)
        df["period_local"] = df["period"].dt.tz_convert(rto_timezones[rto])
        df = df.drop(columns="period").set_index("period_local")
        out_path = f"hourly_fuel_mix/{rto.lower()}_{year}_hourly_local-time.csv"
        df.to_csv(out_path)